In [4]:
pip install numpy pandas matplotlib nltk scikit-learn scipy

#  SECTION 1: IMPORTS AND SETUP

In [7]:
import numpy as np
import pandas as pd
import re
import string
import nltk
import os

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Setup complete.


[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# Section 2: Dataset Loading


In [8]:
from google.colab import drive
drive.mount('/content/drive')

# ── FOLDER STRUCTURE ─────────────────────────────────────────────────────────
# Recommended layout inside your Google Drive:
#
#   MyDrive/
#   └── Reuters21578/
#       ├── datasets/          ← place all CSV files here
#       ├── checkpoints/       ← pickle files (wsd_docs.pkl, etc.)
#       ├── chain_outputs/     ← lexical chain inspection outputs
#       └── logs/              ← optional run logs
#
# Update DATASET_DIR below to match wherever you placed the CSV files.

DATASET_DIR   = "/content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/datasets/"
CHECKPOINT_DIR = "/content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/checkpoints/"
CHAIN_OUT_DIR  = "/content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/chain_outputs/"

# Create output directories if they don't already exist
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(CHAIN_OUT_DIR,  exist_ok=True)

# ── CHECKPOINT PATH ───────────────────────────────────────────────────────────
WSD_CACHE_FILE = os.path.join(CHECKPOINT_DIR, 'wsd_docs.pkl')

print("Drive mounted and paths configured.")
print(f"  Dataset dir   : {DATASET_DIR}")
print(f"  Checkpoint dir: {CHECKPOINT_DIR}")
print(f"  Chain out dir : {CHAIN_OUT_DIR}")
print(f"  WSD cache     : {WSD_CACHE_FILE}")


Mounted at /content/drive
Drive mounted and paths configured.
  Dataset dir   : /content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/datasets/
  Checkpoint dir: /content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/checkpoints/
  Chain out dir : /content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/chain_outputs/
  WSD cache     : /content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/checkpoints/wsd_docs.pkl


In [11]:
# Reuters-21578 comes in three standard splits:
#
#   ModApte  — most widely used in research; train/test boundary is cleanest
#              (only documents explicitly assigned to train OR test are kept).
#              ~9,600 training + ~3,700 test documents.
#
#   ModLewis — retains more borderline documents; slightly larger but noisier.
#
#   ModHayes — preserves even more documents; rarely used in modern work.


# we're using ModApte

In [13]:
import ast

TRAIN_FILE = os.path.join(DATASET_DIR, 'ModApte_train.csv')
TEST_FILE  = os.path.join(DATASET_DIR, 'ModApte_test.csv')

for fpath in [TRAIN_FILE, TEST_FILE]:
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  ✓  Found: {os.path.basename(fpath)}  ({size_mb:.2f} MB)")
    else:
        print(f"  ✗  MISSING: {fpath}")
        print("     → Check that DATASET_DIR is set correctly above.")

print("\nLoading Reuters-21578 ModApte split...")
df_train = pd.read_csv(TRAIN_FILE)
df_test  = pd.read_csv(TEST_FILE)
df_all   = pd.concat([df_train, df_test], ignore_index=True)
print(f"  Train rows : {len(df_train)}")
print(f"  Test rows  : {len(df_test)}")
print(f"  Combined   : {len(df_all)}")

print(f"\nColumns: {list(df_all.columns)}")

def parse_topics(val):
    if isinstance(val, list):
        return val
    if not isinstance(val, str) or val.strip() in ('', '[]', 'nan'):
        return []
    try:
        parsed = ast.literal_eval(val)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

df_all['topics_list'] = df_all['topics'].apply(parse_topics)

df_all['title'] = df_all['title'].fillna('')
df_all['text']  = df_all['text'].fillna('')
df_all['full_text'] = (df_all['title'] + ' ' + df_all['text']).str.strip()

# drop empty docs
df_all = df_all[df_all['full_text'].str.len() > 20].copy()
print(f"\nDocuments with non-empty text: {len(df_all)}")

df_filtered = df_all[df_all['topics_list'].apply(len) > 0].copy()
print(f"Documents with at least one topic: {len(df_filtered)}")

documents = df_filtered['full_text'].tolist()
# Store topic lists alongside for reference (not used in WSD/chains)
doc_topics = df_filtered['topics_list'].tolist()

print(f"\n{'─'*50}")
print(f"FINAL CORPUS SUMMARY")
print(f"{'─'*50}")
print(f"Total documents : {len(documents)}")
print(f"\nSample document (first 500 chars):")
print(documents[0][:500])
print(f"\nCorresponding topics: {doc_topics[0]}")

  ✓  Found: ModApte_train.csv  (8.65 MB)
  ✓  Found: ModApte_test.csv  (2.80 MB)

Loading Reuters-21578 ModApte split...
  Train rows : 9603
  Test rows  : 3299
  Combined   : 12902

Columns: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title']

Documents with non-empty text: 12902
Documents with at least one topic: 10794

──────────────────────────────────────────────────
FINAL CORPUS SUMMARY
──────────────────────────────────────────────────
Total documents : 10794

Sample document (first 500 chars):
BAHIA COCOA REVIEW Showers continued throughout the week in
the Bahia cocoa zone, alleviating the drought since early
January and improving prospects for the coming temporao,
although normal humidity levels have not been restored,
Comissaria Smith said in its weekly review.
    The dry period means the temporao will be late this year.
    Arrivals for the week ended February 22 were 155,221 bags
of 60 k

In [27]:
import ast

TRAIN_FILE = os.path.join(DATASET_DIR, 'ModApte_train.csv')
TEST_FILE  = os.path.join(DATASET_DIR, 'ModApte_test.csv')

for fpath in [TRAIN_FILE, TEST_FILE]:
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  ✓  Found: {os.path.basename(fpath)}  ({size_mb:.2f} MB)")
    else:
        print(f"  ✗  MISSING: {fpath}")
        print("     → Check that DATASET_DIR is set correctly above.")

print("\nLoading Reuters-21578 ModApte split...")
df_train = pd.read_csv(TRAIN_FILE)
df_test  = pd.read_csv(TEST_FILE)
df_all   = pd.concat([df_train, df_test], ignore_index=True)
print(f"  Train rows : {len(df_train)}")
print(f"  Test rows  : {len(df_test)}")
print(f"  Combined   : {len(df_all)}")

print(f"\nColumns: {list(df_all.columns)}")

# The 'topics' column may be stored as a string representation of a list,
# e.g.  "['earn', 'acq']"  — safely parse it into an actual Python list.
def parse_topics(val):
    if isinstance(val, list):
        return val
    if not isinstance(val, str) or val.strip() in ('', '[]', 'nan'):
        return []
    try:
        parsed = ast.literal_eval(val)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

df_all['topics_list'] = df_all['topics'].apply(parse_topics)

df_all['title'] = df_all['title'].fillna('')
df_all['text']  = df_all['text'].fillna('')
df_all['full_text'] = df_all['title'] + ' ' + df_all['text']
df_all['full_text']  = df_all['full_text'].str.strip()

df_all = df_all[df_all['full_text'].str.len() > 20].copy()
print(f"\nDocuments with non-empty text: {len(df_all)}")

from collections import Counter
topic_counter = Counter(t for topics in df_all['topics_list'] for t in topics)
print("\nTop 50 topics by document frequency:")
for topic, count in topic_counter.most_common(50):
    print(f"  {topic:<20} {count}")

TOP_N = 50
CHOSEN_TOPICS = [topic for topic, count in topic_counter.most_common(TOP_N)]
print(f"\nChosen topics: {CHOSEN_TOPICS}")

def single_label_category(topics):
    """Return the single matching category, or None if 0 or >1 match."""
    matches = [t for t in topics if t in CHOSEN_TOPICS]
    return matches[0] if len(matches) == 1 else None

df_all['category'] = df_all['topics_list'].apply(single_label_category)
df_filtered = df_all[df_all['category'].notna()].copy()
print(f"\nDocuments surviving single-label filter: {len(df_filtered)}")
print("Per-category counts:")
print(df_filtered['category'].value_counts())

MAX_PER_CLASS = 100   # WSD is expensiveee!!

sampled_dfs = []
for topic in CHOSEN_TOPICS:
    subset = df_filtered[df_filtered['category'] == topic]
    n_take = min(MAX_PER_CLASS, len(subset))
    sampled_dfs.append(subset.sample(n=n_take, random_state=RANDOM_STATE))

df_sample = pd.concat(sampled_dfs, ignore_index=True).sample(
    frac=1, random_state=RANDOM_STATE
).reset_index(drop=True)

TOPIC_TO_ID   = {topic: i for i, topic in enumerate(CHOSEN_TOPICS)}
documents     = df_sample['full_text'].tolist()
true_labels   = [TOPIC_TO_ID[cat] for cat in df_sample['category']]

print(f"\n{'─'*50}")
print(f"FINAL CORPUS SUMMARY")
print(f"{'─'*50}")
print(f"Topics selected    : {CHOSEN_TOPICS}")
print(f"\nSample document (first 300 chars):")
print(documents[0][:600])
print(f"\nCorresponding label: {CHOSEN_TOPICS[true_labels[0]]}")


  ✓  Found: ModApte_train.csv  (8.65 MB)
  ✓  Found: ModApte_test.csv  (2.80 MB)

Loading Reuters-21578 ModApte split...
  Train rows : 9603
  Test rows  : 3299
  Combined   : 12902

Columns: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title']

Documents with non-empty text: 12902

Top 50 topics by document frequency:
  earn                 3923
  acq                  2292
  crude                374
  trade                326
  money-fx             293
  interest             271
  money-supply         151
  ship                 144
  money-fxinterest     137
  grainwheat           134
  sugar                122
  coffee               112
  gold                 90
  graincorn            86
  money-fxdlr          80
  gnp                  73
  cpi                  71
  cocoa                61
  grain                51
  alum                 50
  reserves             49
  jobs                 49
  crude

In [36]:
print("Number of selected topics:", len(CHOSEN_TOPICS))
print("Documents retained:", len(df_filtered))

Number of selected topics: 50
Documents retained: 9719


# section 3: Preprocessing



In [38]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# nltk data files already downloaded in Section 1

# initialise tools we will be reusing across all steps
stemmer        = PorterStemmer()
STOPWORDS      = set(stopwords.words('english'))  # common English words to remove

# After stemming, words become unrecognisable (e.g. "running" → "run").
# WordNet needs real English words, not stems !!
# So we keep a global dictionary that maps each stem back to the most common original word — used in Steps 4 and 5.
stem_to_word = {}   # e.g.  {"run": "running",  "space": "space"}

# -----------------------------------------------------------------------------
def clean_text(text):
    """
    Remove noise from a raw document string.
      1. Lowercase everything
      2. Remove URLs
      3. Remove email addresses
      4. Remove non-alphabetic characters (numbers, punctuation, etc.)
      5. Collapse multiple spaces into one
    """
    text = text.lower()                            # 1. lowercase
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # 2. remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)          # 3. remove emails
    text = re.sub(r'[^a-z\s]', ' ', text)         # 4. keep only a-z and spaces
    text = re.sub(r'\s+', ' ', text).strip()       # 5. collapse whitespace
    return text


# -----------------------------------------------------------------------------
def tokenize_and_filter(text):
    """
      1. Tokenize
      2. Remove stopwords
      3. Remove very short words (len < 3)
      4. Remove very long words (len > 20)
    """

    tokens = word_tokenize(text)                   # 1. split into words
    tokens = [t for t in tokens
              if t not in STOPWORDS                # 2. drop stopwords
              and 3 <= len(t) <= 20]               # 3 & 4. length filter
    return tokens


# -----------------------------------------------------------------------------
def stem_tokens(tokens):

    # Reduce each token to its root (stem)
    # updates the global stem_to_word mapping so we can recover real words for WordNet lookups later.

    stemmed = []
    for word in tokens:
        stem = stemmer.stem(word)        # e.g. "running" → "run"
        stemmed.append(stem)

        # Stores the shortest real-word form for each stem
        # (shorter originals tend to be the base form)
        if stem not in stem_to_word:
            stem_to_word[stem] = word
        else:
            # prefer the shorter word as the representative
            if len(word) < len(stem_to_word[stem]):
                stem_to_word[stem] = word

    return stemmed

# only need unstemmed tokens!
def preprocess_document(text):
    text   = clean_text(text)
    tokens = tokenize_and_filter(text)
    return tokens


# -----------------------------------------------------------------------------
def preprocess_corpus(documents):
    """
    Apply preprocess_document to every document in the corpus.
    """
    preprocessed = []
    total = len(documents)

    for i, doc in enumerate(documents):
        # Progress update every 100 documents
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"  Preprocessing document {i+1}/{total}...")
        preprocessed.append(preprocess_document(doc))

    return preprocessed


# preprocessing on the full corpus
print("Starting preprocessing...\n")
preprocessed_docs = preprocess_corpus(documents)
print("\nPreprocessing complete.")

print(f"\nTotal documents preprocessed : {len(preprocessed_docs)}")
print(f"Stem-to-word mapping size    : {len(stem_to_word)} stems")
print(f"\nExample — original (first 200 chars):")
print(documents[0][:200])
print(f"\nExample — preprocessed tokens (first 20):")
print(preprocessed_docs[0][:20])

Starting preprocessing...

  Preprocessing document 100/2640...
  Preprocessing document 200/2640...
  Preprocessing document 300/2640...
  Preprocessing document 400/2640...
  Preprocessing document 500/2640...
  Preprocessing document 600/2640...
  Preprocessing document 700/2640...
  Preprocessing document 800/2640...
  Preprocessing document 900/2640...
  Preprocessing document 1000/2640...
  Preprocessing document 1100/2640...
  Preprocessing document 1200/2640...
  Preprocessing document 1300/2640...
  Preprocessing document 1400/2640...
  Preprocessing document 1500/2640...
  Preprocessing document 1600/2640...
  Preprocessing document 1700/2640...
  Preprocessing document 1800/2640...
  Preprocessing document 1900/2640...
  Preprocessing document 2000/2640...
  Preprocessing document 2100/2640...
  Preprocessing document 2200/2640...
  Preprocessing document 2300/2640...
  Preprocessing document 2400/2640...
  Preprocessing document 2500/2640...
  Preprocessing document 2600/26

# SECTION 4: NOUN EXTRACTION FUNCTIONS

In [41]:
from nltk import pos_tag

# Noun POS tags
NOUN_TAGS = {'NN', 'NNS', 'NNP', 'NNPS'}


def extract_nouns_from_doc(token_list):
    tagged = pos_tag(token_list)

    nouns = [
        word
        for word, tag in tagged
        if tag in NOUN_TAGS
    ]

    return nouns


def extract_nouns_from_corpus(preprocessed_docs):
    """
    Apply noun extraction to every document.
    """
    noun_docs = []
    total = len(preprocessed_docs)

    for i, token_list in enumerate(preprocessed_docs):
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"  Extracting nouns from document {i+1}/{total}...")

        nouns = extract_nouns_from_doc(token_list)
        noun_docs.append(nouns)

    return noun_docs


print("Starting noun extraction...\n")

noun_docs = extract_nouns_from_corpus(preprocessed_docs)

print("\nNoun extraction complete.")

print(f"\nTotal documents processed : {len(noun_docs)}")

total_tokens = sum(len(doc) for doc in preprocessed_docs)
total_nouns  = sum(len(doc) for doc in noun_docs)

print(f"Total tokens             : {total_tokens}")
print(f"Total nouns kept         : {total_nouns}")
print(f"Noun retention rate      : {100 * total_nouns / total_tokens:.1f}%")

print(f"\nExample — tokens (first 20):")
print(preprocessed_docs[0][:20])

print(f"\nExample — nouns (first 20):")
print(noun_docs[0][:20])

Starting noun extraction...

  Extracting nouns from document 100/2640...
  Extracting nouns from document 200/2640...
  Extracting nouns from document 300/2640...
  Extracting nouns from document 400/2640...
  Extracting nouns from document 500/2640...
  Extracting nouns from document 600/2640...
  Extracting nouns from document 700/2640...
  Extracting nouns from document 800/2640...
  Extracting nouns from document 900/2640...
  Extracting nouns from document 1000/2640...
  Extracting nouns from document 1100/2640...
  Extracting nouns from document 1200/2640...
  Extracting nouns from document 1300/2640...
  Extracting nouns from document 1400/2640...
  Extracting nouns from document 1500/2640...
  Extracting nouns from document 1600/2640...
  Extracting nouns from document 1700/2640...
  Extracting nouns from document 1800/2640...
  Extracting nouns from document 1900/2640...
  Extracting nouns from document 2000/2640...
  Extracting nouns from document 2100/2640...
  Extracting n

# SECTION 5: WORD SENSE DISAMBIGUATION (WSD)


In [43]:
import os
import pickle

from nltk.corpus import wordnet as wn

try:
    from tqdm import tqdm
    TQDM_AVAILABLE = True
except ImportError:
    TQDM_AVAILABLE = False

MAX_SENSES = 5

_synset_cache = {}
_similarity_cache = {}


def get_top_senses(word):
    if word not in _synset_cache:
        synsets = wn.synsets(word, pos=wn.NOUN)
        _synset_cache[word] = synsets[:MAX_SENSES]
    return _synset_cache[word]


def wu_palmer_similarity(synset_a, synset_b):
    key = frozenset([synset_a.name(), synset_b.name()])

    if key not in _similarity_cache:
        score = synset_a.wup_similarity(synset_b)
        _similarity_cache[key] = score if score is not None else 0.0

    return _similarity_cache[key]


def disambiguate_word(target_word, context_words_deduped):
    target_senses = get_top_senses(target_word)

    if not target_senses:
        return None

    if len(target_senses) == 1:
        return target_senses[0]

    best_sense = None
    best_score = -1.0

    for candidate in target_senses:
        total_score = 0.0

        for ctx_word in context_words_deduped:
            ctx_senses = get_top_senses(ctx_word)

            if not ctx_senses:
                continue

            max_sim = max(
                wu_palmer_similarity(candidate, ctx_sense)
                for ctx_sense in ctx_senses
            )

            total_score += max_sim

        if total_score > best_score:
            best_score = total_score
            best_sense = candidate

    return best_sense


def disambiguate_document(nouns):
    all_words_deduped = list(dict.fromkeys(nouns))

    result = []

    for word in nouns:
        context = [w for w in all_words_deduped if w != word]

        best_sense = disambiguate_word(word, context)

        result.append((word, best_sense))

    return result


def disambiguate_corpus(noun_docs):
    wsd_results = []
    total = len(noun_docs)

    iterator = (
        tqdm(noun_docs, desc="WSD Progress", unit="doc")
        if TQDM_AVAILABLE
        else noun_docs
    )

    for i, nouns in enumerate(iterator):
        if not TQDM_AVAILABLE and ((i + 1) % 50 == 0 or (i + 1) == total):
            print(
                f"  WSD: document {i+1}/{total}  "
                f"| synset cache: {len(_synset_cache)} words  "
                f"| sim cache: {len(_similarity_cache)} pairs"
            )

        wsd_results.append(
            disambiguate_document(nouns)
        )

    return wsd_results


if os.path.exists(WSD_CACHE_FILE):
    print("Loading cached WSD results...")

    with open(WSD_CACHE_FILE, 'rb') as f:
        wsd_docs_names = pickle.load(f)

    wsd_docs = [
        [
            (
                word,
                wn.synset(name) if name is not None else None
            )
            for (word, name) in doc
        ]
        for doc in wsd_docs_names
    ]

    print("WSD complete.")

else:
    print("Running WSD...")
    print("(First run takes several minutes — subsequent runs load instantly)\n")

    wsd_docs = disambiguate_corpus(noun_docs)

    wsd_docs_names = [
        [
            (
                word,
                syn.name() if syn is not None else None
            )
            for (word, syn) in doc
        ]
        for doc in wsd_docs
    ]

    print("\nSaving WSD results...")

    with open(WSD_CACHE_FILE, 'wb') as f:
        pickle.dump(wsd_docs_names, f)

    print("WSD complete.")


total_nouns = sum(len(doc) for doc in wsd_docs)

matched_nouns = sum(
    1
    for doc in wsd_docs
    for (_, syn) in doc
    if syn is not None
)

print(f"\nTotal noun tokens  : {total_nouns}")
print(f"Matched to WordNet : {matched_nouns}")
print(f"WordNet coverage   : {100 * matched_nouns / total_nouns:.1f}%")
print(f"Synset cache size  : {len(_synset_cache)} unique words")
print(f"Sim cache size     : {len(_similarity_cache)} unique pairs")

print("\nExample — WSD output for first document (first 5 words):")

for word, syn in wsd_docs[0][:5]:
    if syn:
        print(
            f"  word='{word:<15}' "
            f"synset='{syn.name():<22}' "
            f"def: '{syn.definition()[:55]}...'"
        )
    else:
        print(
            f"  word='{word:<15}' "
            f"NOT FOUND in WordNet"
        )

Running WSD...
(First run takes several minutes — subsequent runs load instantly)



WSD Progress: 100%|██████████| 2640/2640 [18:44<00:00,  2.35doc/s]



Saving WSD results...
WSD complete.

Total noun tokens  : 129082
Matched to WordNet : 114847
WordNet coverage   : 89.0%
Synset cache size  : 8747 unique words
Sim cache size     : 5426937 unique pairs

Example — WSD output for first document (first 5 words):
  word='mobil          ' NOT FOUND in WordNet
  word='mob            ' synset='syndicate.n.01        ' def: 'a loose affiliation of gangsters in charge of organized...'
  word='capital        ' synset='capital.n.01          ' def: 'assets available for use in the production of further a...'
  word='mobil          ' NOT FOUND in WordNet
  word='corp           ' synset='corporation.n.01      ' def: 'a business firm whose articles of incorporation have be...'


In [44]:

print("\nExample — WSD output for first document (first 20 words):")

for word, syn in wsd_docs[0][:20]:
    if syn:
        print(
            f"  word='{word:<15}' "
            f"synset='{syn.name():<22}' "
            f"def: '{syn.definition()[:55]}...'"
        )
    else:
        print(
            f"  word='{word:<15}' "
            f"NOT FOUND in WordNet"
        )


Example — WSD output for first document (first 20 words):
  word='mobil          ' NOT FOUND in WordNet
  word='mob            ' synset='syndicate.n.01        ' def: 'a loose affiliation of gangsters in charge of organized...'
  word='capital        ' synset='capital.n.01          ' def: 'assets available for use in the production of further a...'
  word='mobil          ' NOT FOUND in WordNet
  word='corp           ' synset='corporation.n.01      ' def: 'a business firm whose articles of incorporation have be...'
  word='chairman       ' synset='president.n.04        ' def: 'the officer who presides at the meetings of an organiza...'
  word='allen          ' synset='allen.n.02            ' def: 'United States filmmaker and comic actor (1935-)...'
  word='murray         ' synset='murray.n.02           ' def: 'Scottish philologist and the lexicographer who shaped t...'
  word='report         ' synset='report.n.02           ' def: 'the act of informing by verbal report...'
  word='today 

# SECTION 6: LEXICAL CHAIN FUNCTIONS

In [45]:
# ============================================================
# SECTION 6: LEXICAL CHAIN FUNCTIONS  (final version)
# ============================================================

WEIGHT_SYNONYM  = 0.8
WEIGHT_HYPERNYM = 0.5
WEIGHT_MERONYM  = 0.3

# Depth 3 gives a good balance:
#   depth 1 → all singletons (too strict)
#   depth 2 → 81% singletons (still too strict)
#   depth 3 → captures domain-level groupings (space, medicine, sports etc.)
#   depth 4+ → over-merges unrelated concepts via distant WordNet root
RELATION_DEPTH = 3

# Wu-Palmer similarity threshold for LCS fallback.
# Two synsets that are not directly related by hypernym/meronym
# but have WP similarity above this threshold are still considered
# related for chaining purposes.
# 0.5 is conservative enough to avoid false merges.
WP_FALLBACK_THRESHOLD = 0.5


# -----------------------------------------------------------------------------
def get_ancestors(synset, depth):
    """
    Collect all hypernym synsets reachable within `depth` hops upward.
    Identical to previous version.
    """
    visited = set()
    frontier = [synset]
    for _ in range(depth):
        next_frontier = []
        for s in frontier:
            for h in s.hypernyms():
                if h not in visited:
                    visited.add(h)
                    next_frontier.append(h)
        frontier = next_frontier
    return visited


# -----------------------------------------------------------------------------
def get_meronyms(synset, depth):
    """
    Collect all meronym synsets reachable within `depth` hops.
    Identical to previous version.
    """
    def direct_meronyms(s):
        return (s.part_meronyms() +
                s.substance_meronyms() +
                s.member_meronyms())

    visited = set()
    frontier = [synset]
    for _ in range(depth):
        next_frontier = []
        for s in frontier:
            for m in direct_meronyms(s):
                if m not in visited:
                    visited.add(m)
                    next_frontier.append(m)
        frontier = next_frontier
    return visited


# -----------------------------------------------------------------------------
def get_relation(synset_a, synset_b):
    """
    Determine whether two synsets share a semantic relation and
    return the corresponding weight. Checks in priority order:

      1. Identity   — same synset
      2. Synonym    — shared lemma name
      3. Hypernym   — ancestor or shared ancestor within RELATION_DEPTH hops
      4. Meronym    — part-of relation within RELATION_DEPTH hops
      5. WP fallback— Wu-Palmer similarity above WP_FALLBACK_THRESHOLD
                      catches domain-related concepts that sit too far
                      apart for structural checks alone

    Input  : two Synset objects
    Output : float weight if related, None otherwise
    """

    # identity
    if synset_a == synset_b:
        return WEIGHT_SYNONYM

    # synonym
    lemmas_a = set(l.name() for l in synset_a.lemmas())
    lemmas_b = set(l.name() for l in synset_b.lemmas())
    if lemmas_a & lemmas_b:
        return WEIGHT_SYNONYM

    # Hypernym / shared ancestor within RELATION_DEPTH
    ancestors_a = get_ancestors(synset_a, RELATION_DEPTH)
    ancestors_b = get_ancestors(synset_b, RELATION_DEPTH)

    if synset_b in ancestors_a or synset_a in ancestors_b:
        return WEIGHT_HYPERNYM

    if ancestors_a & ancestors_b:
        return WEIGHT_HYPERNYM

    # Meronym within RELATION_DEPTH
    meronyms_a = get_meronyms(synset_a, RELATION_DEPTH)
    meronyms_b = get_meronyms(synset_b, RELATION_DEPTH)

    if synset_b in meronyms_a or synset_a in meronyms_b:
        return WEIGHT_MERONYM

    if meronyms_a & meronyms_b:
        return WEIGHT_MERONYM

    # Wu-Palmer similarity fallback
    # adds no extra WordNet traversal cost for pairs already seen in WSD.
    wp_score = wu_palmer_similarity(synset_a, synset_b)
    if wp_score >= WP_FALLBACK_THRESHOLD:
        return WEIGHT_HYPERNYM   # treat as hypernym-strength relation

    return None


# -----------------------------------------------------------------------------
def build_lexical_chains(wsd_pairs):
    valid_pairs = [(stem, syn) for (stem, syn) in wsd_pairs
                   if syn is not None]
    if not valid_pairs:
        return []

    # Concept weights (term frequency, summed for synonyms)
    synset_weight = {}
    synset_stem   = {}
    for stem, syn in valid_pairs:
        name = syn.name()
        synset_weight[name] = synset_weight.get(name, 0) + 1
        if name not in synset_stem:
            synset_stem[name] = stem

    # Deduplicated unique concepts
    seen = {}
    for _, syn in valid_pairs:
        seen[syn.name()] = syn
    unique_pairs = [(synset_stem[name], syn) for name, syn in seen.items()]

    n = len(unique_pairs)

    # Union-Find
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry

    for i in range(n):
        for j in range(i + 1, n):
            if find(i) != find(j):
                _, syn_i = unique_pairs[i]
                _, syn_j = unique_pairs[j]
                if get_relation(syn_i, syn_j) is not None:
                    union(i, j)

    # Group by chain root
    chains_dict = {}
    for i, (stem, syn) in enumerate(unique_pairs):
        root = find(i)
        if root not in chains_dict:
            chains_dict[root] = []
        chains_dict[root].append((stem, syn, synset_weight[syn.name()]))

    return list(chains_dict.values())


# -----------------------------------------------------------------------------
def build_chains_for_corpus(wsd_docs):

    # Apply build_lexical_chains to every document in the corpus.
    chain_docs = []
    total = len(wsd_docs)
    for i, wsd_pairs in enumerate(wsd_docs):
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"  Building chains: document {i+1}/{total}...")
        chain_docs.append(build_lexical_chains(wsd_pairs))
    return chain_docs


print("Building lexical chains (depth=3 + WP fallback)...\n")
chain_docs = build_chains_for_corpus(wsd_docs)
print("\nLexical chain construction complete.")

total_chains   = sum(len(chains) for chains in chain_docs)
total_concepts = sum(len(c) for chains in chain_docs for c in chains)
chain_lengths  = [len(c) for chains in chain_docs for c in chains]

avg_chains_per_doc = total_chains / len(chain_docs)
avg_chain_length   = total_concepts / total_chains if total_chains else 0
singleton_count    = sum(1 for l in chain_lengths if l == 1)

print(f"\nTotal documents      : {len(chain_docs)}")
print(f"Total chains         : {total_chains}")
print(f"Avg chains / doc     : {avg_chains_per_doc:.1f}")
print(f"Total concepts       : {total_concepts}")
print(f"Avg concepts / chain : {avg_chain_length:.1f}")
print(f"Longest chain        : {max(chain_lengths)} concepts")
print(f"Single-node chains   : {singleton_count} "
      f"({100 * singleton_count / total_chains:.1f}%)")

# largest chains for first document
print(f"\nExample — top 3 largest chains for first document:")
sorted_chains = sorted(chain_docs[0], key=lambda c: len(c), reverse=True)
for idx, chain in enumerate(sorted_chains[:3]):
    print(f"\n  Chain {idx+1} ({len(chain)} concept(s)):")
    for stem, syn, weight in chain[:5]:   # show max 5 concepts per chain
        print(f"    stem='{stem:<12}'  synset='{syn.name():<25}'"
              f"  w={weight}  '{syn.definition()[:45]}...'")
    if len(chain) > 5:
        print(f"    ... and {len(chain)-5} more concepts")

Building lexical chains (depth=3 + WP fallback)...

  Building chains: document 100/2640...
  Building chains: document 200/2640...
  Building chains: document 300/2640...
  Building chains: document 400/2640...
  Building chains: document 500/2640...
  Building chains: document 600/2640...
  Building chains: document 700/2640...
  Building chains: document 800/2640...
  Building chains: document 900/2640...
  Building chains: document 1000/2640...
  Building chains: document 1100/2640...
  Building chains: document 1200/2640...
  Building chains: document 1300/2640...
  Building chains: document 1400/2640...
  Building chains: document 1500/2640...
  Building chains: document 1600/2640...
  Building chains: document 1700/2640...
  Building chains: document 1800/2640...
  Building chains: document 1900/2640...
  Building chains: document 2000/2640...
  Building chains: document 2100/2640...
  Building chains: document 2200/2640...
  Building chains: document 2300/2640...
  Building cha

In [47]:
# check to make sure chains are right
def print_document_chains(doc_id,
                          max_chains=10,
                          max_concepts_per_chain=20):

    if doc_id >= len(chain_docs):
        print(f"Document {doc_id} does not exist.")
        return

    chains = sorted(
        chain_docs[doc_id],
        key=lambda c: len(c),
        reverse=True
    )

    print("=" * 80)
    print(f"DOCUMENT {doc_id}")
    print(f"Total chains: {len(chains)}")
    print("=" * 80)

    for chain_no, chain in enumerate(chains[:max_chains], start=1):

        print(f"\nCHAIN {chain_no}")
        print(f"Size: {len(chain)} concepts")
        print("-" * 60)

        for stem, syn, weight in chain[:max_concepts_per_chain]:
            print(
                f"{stem:<15}"
                f"{syn.name():<30}"
                f"freq={weight:<3}"
                f"{syn.definition()}"
            )

        if len(chain) > max_concepts_per_chain:
            print(f"... {len(chain)-max_concepts_per_chain} more concepts")



docs_to_check = [0, 1, 2, 3, 4]

for doc_id in range(10):
    print_document_chains(
        doc_id,
        max_chains=5,
        max_concepts_per_chain=15
    )
    print("\n\n")

DOCUMENT 0
Total chains: 5

CHAIN 1
Size: 59 concepts
------------------------------------------------------------
mob            syndicate.n.01                freq=1  a loose affiliation of gangsters in charge of organized criminal activities
capital        capital.n.01                  freq=4  assets available for use in the production of further assets
corp           corporation.n.01              freq=2  a business firm whose articles of incorporation have been approved in some state
chairman       president.n.04                freq=2  the officer who presides at the meetings of an organization
allen          allen.n.02                    freq=1  United States filmmaker and comic actor (1935-)
murray         murray.n.02                   freq=4  Scottish philologist and the lexicographer who shaped the Oxford English Dictionary (1837-1915)
report         report.n.02                   freq=3  the act of informing by verbal report
today          today.n.01                    freq=1  t